Import libraries and load interaction data

In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# Load Task 7 interaction dataset
file_path = "/content/07_user_interaction_data.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(
        "Task 7 interaction CSV not found. Check the file path."
    )

interactions = pd.read_csv(file_path)

print("SIMULATED DATA - NOT REAL USER ACTIVITY")

# Display first five rows
print("\nFirst five rows:")
display(interactions.head())

# Display dataset shape
print("\nDataset Shape:", interactions.shape)

# Display column names
print("\nColumn Names:")
print(interactions.columns.tolist())

# Display data types
print("\nData Types:")
print(interactions.dtypes)

# Display missing values
print("\nMissing Values:")
print(interactions.isnull().sum())

# Check required columns
required_columns = ["user_id", "ted_talk_id", "rating"]

for column in required_columns:
    if column not in interactions.columns:
        raise ValueError(column + " column is missing!")

print("\nRequired columns verified!")

SIMULATED DATA - NOT REAL USER ACTIVITY

First five rows:


,user_id,ted_talk_id,talk_title,rating,liked,watched,saved
0,U001,TED_SAMPLE_001,Do schools kill creativity?,5.0,True,True,True
1,U001,TED_SAMPLE_002,Averting the climate crisis,4.0,True,True,True
2,U001,TED_SAMPLE_003,Simplicity sells,5.0,True,True,True
3,U001,TED_SAMPLE_004,Greening the ghetto,2.0,False,True,False
4,U001,TED_SAMPLE_005,The best stats you've ever seen,1.0,False,True,False



Dataset Shape: (29, 7)

Column Names:
['user_id', 'ted_talk_id', 'talk_title', 'rating', 'liked', 'watched', 'saved']

Data Types:
user_id         object
ted_talk_id     object
talk_title      object
rating         float64
liked             bool
watched           bool
saved             bool
dtype: object

Missing Values:
user_id        0
ted_talk_id    0
talk_title     0
rating         1
liked          0
watched        0
saved          0
dtype: int64

Required columns verified!


Validate data and build the User-Item Matrix

Before creating the matrix, we must check duplicate user-talk pairs and invalid ratings.
We will preserve the original missing values as NaN

In [ ]:
# Convert IDs into strings
interactions["user_id"] = (
    interactions["user_id"].astype(str).str.strip()
)

interactions["ted_talk_id"] = (
    interactions["ted_talk_id"].astype(str).str.strip()
)

# Check missing IDs
if interactions["user_id"].isin(["", "nan"]).any():
    raise ValueError("Missing user IDs found!")

if interactions["ted_talk_id"].isin(["", "nan"]).any():
    raise ValueError("Missing TED Talk IDs found!")

# Check duplicate user-talk pairs
duplicates = interactions.duplicated(
    subset=["user_id", "ted_talk_id"]
)

print("Duplicate user-talk pairs:", duplicates.sum())

if duplicates.any():
    display(interactions[duplicates])
    raise ValueError("Duplicate user-talk records must be corrected.")

# Convert ratings into numeric values
original_rating = interactions["rating"].copy()

interactions["rating"] = pd.to_numeric(
    interactions["rating"],
    errors="coerce"
)

# Detect invalid non-empty ratings
invalid_text = (
    original_rating.notna() &
    original_rating.astype(str).str.strip().ne("") &
    interactions["rating"].isna()
)

invalid_range = (
    interactions["rating"].notna() &
    ~interactions["rating"].between(1, 5)
)

if invalid_text.any() or invalid_range.any():
    raise ValueError("Invalid ratings found. Ratings must be between 1 and 5.")

print("\nRating validation successful!")

# Create original User-Item Matrix
user_item_matrix = interactions.pivot(
    index="user_id",
    columns="ted_talk_id",
    values="rating"
)

print("\nORIGINAL USER-ITEM MATRIX")

display(user_item_matrix)

print("Matrix Shape:", user_item_matrix.shape)

Duplicate user-talk pairs: 0

Rating validation successful!

ORIGINAL USER-ITEM MATRIX


ted_talk_id,TED_SAMPLE_001,TED_SAMPLE_002,TED_SAMPLE_003,TED_SAMPLE_004,TED_SAMPLE_005,TED_SAMPLE_006
user_id,,,,,,
U001,5.0,4.0,5.0,2.0,1.0,NaN
U002,5.0,4.0,4.0,2.0,1.0,5.0
U003,1.0,2.0,1.0,5.0,5.0,1.0
U004,4.0,5.0,4.0,1.0,2.0,NaN
U005,2.0,1.0,2.0,5.0,4.0,1.0


Matrix Shape: (5, 6)


Create a separate matrix and calculate user similarity ,Cosine similarity requires numerical vectors.

Therefore, we create a separate copy of the original matrix and replace missing ratings with zero.
Important distinction
Here, zero is only a computational placeholder. It does not mean that the user gave the TED Talk a rating of zero or disliked it.

In [ ]:
# Create a separate copy
# Fill missing values with zero only for calculation
filled_matrix = user_item_matrix.fillna(0)

print("ZERO-FILLED MATRIX FOR CALCULATION")

display(filled_matrix)

# Calculate cosine similarity between users
similarity_values = cosine_similarity(filled_matrix)

# Convert similarity values into a dataframe
similarity_matrix = pd.DataFrame(
    similarity_values,
    index=filled_matrix.index,
    columns=filled_matrix.index
)

print("\nUSER SIMILARITY MATRIX")

display(similarity_matrix.round(3))

ZERO-FILLED MATRIX FOR CALCULATION


ted_talk_id,TED_SAMPLE_001,TED_SAMPLE_002,TED_SAMPLE_003,TED_SAMPLE_004,TED_SAMPLE_005,TED_SAMPLE_006
user_id,,,,,,
U001,5.0,4.0,5.0,2.0,1.0,0.0
U002,5.0,4.0,4.0,2.0,1.0,5.0
U003,1.0,2.0,1.0,5.0,5.0,1.0
U004,4.0,5.0,4.0,1.0,2.0,0.0
U005,2.0,1.0,2.0,5.0,4.0,1.0



USER SIMILARITY MATRIX


user_id,U001,U002,U003,U004,U005
user_id,,,,,
U001,1.000,0.840,0.519,0.965,0.631
U002,0.840,1.000,0.525,0.817,0.616
U003,0.519,0.525,1.000,0.555,0.964
U004,0.965,0.817,0.555,1.000,0.605
U005,0.631,0.616,0.964,0.605,1.000


Find similar users
Now we select a target user and rank the other users based on similarity.

The target user must be excluded from the similar-user list.

In [ ]:
# Select sample user
target_user = "U001"

# Check whether user exists
if target_user not in similarity_matrix.index:
    raise ValueError("User ID not found!")

# Get similarity scores for selected user
similar_users = similarity_matrix.loc[target_user]

# Remove target user
similar_users = similar_users.drop(target_user)

# Sort users by similarity
similar_users = similar_users.sort_values(
    ascending=False
)

print("TARGET USER:", target_user)

print("\nSIMILAR USERS")

display(
    similar_users.reset_index().rename(
        columns={
            "index": "Similar User",
            target_user: "Similarity Score"
        }
    )
)

TARGET USER: U001

SIMILAR USERS


,user_id,Similarity Score
0,U004,0.964617
1,U002,0.839760
2,U005,0.631494
3,U003,0.518737


Build the recommendation function
Main Task 8 requirement
We now create the required function:

recommend_for_user(user_id, top_n=5)

It performs five operations:
1. Validate the given user ID.
2. Find similar users.
3. Identify TED Talks the target user has not rated.
4. Calculate predicted ratings using a weighted average.
5. Return the highest-scoring recommendations.

In [ ]:
def recommend_for_user(user_id, top_n=5):

    # Step 1: Validate user ID
    if user_id not in user_item_matrix.index:
        raise ValueError("Unknown User ID: " + str(user_id))

    if not isinstance(top_n, int) or top_n <= 0:
        raise ValueError("top_n must be a positive integer.")

    # Step 2: Get ratings of target user
    user_ratings = user_item_matrix.loc[user_id]

    # Step 3: Find similar users
    similar_users = similarity_matrix.loc[user_id]

    # Exclude target user
    similar_users = similar_users.drop(user_id)

    # Sort by similarity
    similar_users = similar_users.sort_values(
        ascending=False
    )

    # Step 4: Find talks not rated by target user
    unseen_talks = user_ratings[
        user_ratings.isna()
    ].index

    recommendations = []

    # Step 5: Predict rating for each unseen talk
    for talk_id in unseen_talks:

        weighted_total = 0
        similarity_total = 0

        # Check ratings from similar users
        for other_user in similar_users.index:

            similarity = similar_users[other_user]

            # Use only positive similarities
            if similarity <= 0:
                continue

            other_rating = user_item_matrix.loc[
                other_user, talk_id
            ]

            # Skip if other user has not rated this talk
            if pd.isna(other_rating):
                continue

            # Calculate weighted rating
            weighted_total += similarity * other_rating

            similarity_total += similarity

        # Avoid division by zero
        if similarity_total > 0:

            predicted_rating = (
                weighted_total / similarity_total
            )

            recommendations.append({

                "ted_talk_id": talk_id,

                "predicted_rating": round(
                    predicted_rating, 3
                )
            })

    # Step 6: Convert results into dataframe
    recommendation_df = pd.DataFrame(
        recommendations,
        columns=["ted_talk_id", "predicted_rating"]
    )

    # Step 7: Sort highest predicted ratings first
    recommendation_df = recommendation_df.sort_values(
        by="predicted_rating",
        ascending=False
    )

    # Step 8: Return top N recommendations
    return recommendation_df.head(top_n).reset_index(drop=True)


print("Recommendation function created successfully!")

Recommendation function created successfully!


Test recommendations for three users and an invalid user
We need to test at least three valid users and one invalid user.

For the Task 7 sample data, U001 and U004 have unrated talks. U002 has rated all six talks, so it may correctly receive an empty recommendation list.

This is useful evidence that the function handles different user histories.

In [ ]:
# Select three valid users
test_users = ["U001", "U004", "U002"]

all_recommendations = {}

for user in test_users:

    print("\n" + "=" * 50)

    print("RECOMMENDATIONS FOR:", user)

    # Generate recommendations
    recommendations = recommend_for_user(
        user_id=user,
        top_n=5
    )

    all_recommendations[user] = recommendations

    if recommendations.empty:

        print("No unseen talks available for recommendation.")

    else:

        display(recommendations)

        # Check that predicted scores are descending
        assert recommendations[
            "predicted_rating"
        ].is_monotonic_decreasing

        # Check that already rated talks are excluded
        rated_talks = user_item_matrix.loc[user].dropna().index

        recommended_talks = recommendations[
            "ted_talk_id"
        ].tolist()

        assert not set(rated_talks).intersection(
            recommended_talks
        )

        print("Recommendation validation successful!")


# Test invalid user
print("\nINVALID USER TEST")

try:

    recommend_for_user("U999")

except ValueError as error:

    print("Correctly handled:", error)


RECOMMENDATIONS FOR: U001


,ted_talk_id,predicted_rating
0,TED_SAMPLE_006,2.688


Recommendation validation successful!

RECOMMENDATIONS FOR: U004


,ted_talk_id,predicted_rating
0,TED_SAMPLE_006,2.653


Recommendation validation successful!

RECOMMENDATIONS FOR: U002
No unseen talks available for recommendation.

INVALID USER TEST
Correctly handled: Unknown User ID: U999


Map recommended TED Talk IDs to titles
This is an important requirement.

We must compare interaction dataset IDs against the original cleaned TED Talks dataset IDs. If an ID cannot be matched, we will report it instead of silently showing a missing title.

The code also supports the talk_title column saved in your Task 7 CSV as a fallback, clearly marking that title as unverified against the cleaned dataset.

In [ ]:
# Load original cleaned TED Talks dataset
cleaned_file = "/content/07_user_interaction_data.csv"

if not os.path.exists(cleaned_file):
    raise FileNotFoundError("Cleaned TED Talks dataset not found!")

cleaned_df = pd.read_csv(cleaned_file)

# Identify original dataset columns
cleaned_columns = {
    c.lower().strip(): c
    for c in cleaned_df.columns
}

# Find title column with fallbacks
title_column = None
for name in ["title", "talk_title", "talktitle"]:
    if name in cleaned_columns:
        title_column = cleaned_columns[name]
        break

if title_column is None:
    raise ValueError("Title column not found in cleaned dataset!")

# Find TED Talk ID column
id_column = None

for name in ["ted_talk_id", "talk_id", "id", "talkid"]:

    if name in cleaned_columns:
        id_column = cleaned_columns[name]
        break


# Standardize IDs for matching
def clean_id(value):

    if pd.isna(value):
        return None

    value = str(value).strip()

    if value.endswith(".0") and value[:-2].isdigit():
        value = value[:-2]

    return value


# Create ID-to-title mapping from original dataset
cleaned_id_map = {}

if id_column is not None:

    for _, row in cleaned_df.iterrows():

        talk_id = clean_id(row[id_column])

        if talk_id is not None:
            cleaned_id_map[talk_id] = row[title_column]


# Check IDs in interaction dataset
interaction_ids = set(
    interactions["ted_talk_id"].map(clean_id)
)

unmatched_ids = []

for talk_id in interaction_ids:

    if talk_id not in cleaned_id_map:
        unmatched_ids.append(talk_id)


print("TED TALK ID MATCHING")

if id_column is None:

    print(
        "No supported TED Talk ID column found in cleaned dataset."
    )

else:

    print("Matched IDs:", len(interaction_ids) - len(unmatched_ids))
    print("Unmatched IDs:", len(unmatched_ids))

    if unmatched_ids:
        print("\nUnmatched TED Talk IDs:")
        print(unmatched_ids)


# Create fallback title mapping from Task 7 CSV
fallback_title_map = {}

if "talk_title" in interactions.columns:

    fallback_title_map = (
        interactions.drop_duplicates("ted_talk_id")
        .set_index("ted_talk_id")["talk_title"]
        .to_dict()
    )


# Combine recommendations from tested users
final_results = []

for user, recommendations in all_recommendations.items():

    for _, row in recommendations.iterrows():

        talk_id = row["ted_talk_id"]

        normalized_id = clean_id(talk_id)

        if normalized_id in cleaned_id_map:

            title = cleaned_id_map[normalized_id]

            title_source = "Cleaned dataset - ID matched"

        elif talk_id in fallback_title_map:

            title = fallback_title_map[talk_id]

            title_source = "Task 7 CSV - ID not verified"

        else:

            title = "Title not found"

            title_source = "Unmatched ID"

        final_results.append({

            "User ID": user,

            "TED Talk ID": talk_id,

            "TED Talk Title": title,

            "Predicted Rating": row["predicted_rating"],

            "Title Source": title_source
        })


final_results_df = pd.DataFrame(
    final_results,
    columns=[
        "User ID",
        "TED Talk ID",
        "TED Talk Title",
        "Predicted Rating",
        "Title Source"
    ]
)

print("\nFINAL PERSONALIZED RECOMMENDATIONS")

display(final_results_df)

TED TALK ID MATCHING
Matched IDs: 6
Unmatched IDs: 0

FINAL PERSONALIZED RECOMMENDATIONS


,User ID,TED Talk ID,TED Talk Title,Predicted Rating,Title Source
0,U001,TED_SAMPLE_006,Why we do what we do,2.688,Cleaned dataset - ID matched
1,U004,TED_SAMPLE_006,Why we do what we do,2.653,Cleaned dataset - ID matched


– Generate observations and the PDF report
This cell generates the required report:

reports/task_08_observations.pdf

It includes the method, similarity findings, test-user results, recommendations, ID-matching findings, limitations and conclusion

In [ ]:

import os
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table
from reportlab.lib.styles import getSampleStyleSheet
from xml.sax.saxutils import escape

# 1. Create reports folder
os.makedirs("reports", exist_ok=True)

# 2. Create PDF
pdf_path = "reports/task_08_observations.pdf"
pdf = SimpleDocTemplate(pdf_path, pagesize=A4)

styles = getSampleStyleSheet()
content = []


# 3. Function to add a heading
def add_heading(text):
    content.append(Paragraph(text, styles["Heading2"]))
    content.append(Spacer(1, 10))


# 4. Function to add normal text
def add_text(text):
    content.append(Paragraph(text, styles["BodyText"]))
    content.append(Spacer(1, 10))


# 5. Function to create a table
def add_table(data):
    table = Table(data)

    table.setStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.black),
        ("PADDING", (0, 0), (-1, -1), 5)
    ])

    content.append(table)
    content.append(Spacer(1, 10))


# 6. PDF title
content.append(Paragraph(
    "Task 8 - Building a Collaborative Filtering Model",
    styles["Title"]
))
content.append(Spacer(1, 15))


# 7. Method
add_heading("1. Method")

add_text(
    "This project uses user-based collaborative filtering. "
    "A User-Item Matrix is created using user ratings. "
    "Missing ratings are kept as NaN in the original matrix. "
    "A separate copy with zeros is used for cosine similarity."
)


# 8. Dataset Summary
add_heading("2. Dataset Summary")

dataset_table = [
    ["Description", "Value"],
    ["Users", str(user_item_matrix.shape[0])],
    ["TED Talks", str(user_item_matrix.shape[1])],
    ["Recorded interactions", str(len(interactions))],
    ["Similarity method", "Cosine Similarity"],
    ["Prediction method", "Weighted Average"]
]

add_table(dataset_table)


# 9. Similarity Matrix
add_heading("3. User Similarity Matrix")

similarity_table = [
    ["User"] + [str(user) for user in similarity_matrix.columns]
]

for user in similarity_matrix.index:
    row = [str(user)]

    for value in similarity_matrix.loc[user]:
        row.append(str(round(value, 3)))

    similarity_table.append(row)

add_table(similarity_table)


# 10. Recommendation Results
add_heading("4. Recommendation Results")

for user, recommendations in all_recommendations.items():

    content.append(Paragraph(
        "User: " + escape(str(user)),
        styles["Heading3"]
    ))

    if recommendations.empty:
        add_text("No recommendations available for this user.")

    else:
        rec_table = [["TED Talk ID", "Predicted Rating"]]

        for _, row in recommendations.iterrows():
            rec_table.append([
                escape(str(row["ted_talk_id"])),
                str(round(row["predicted_rating"], 2))
            ])

        add_table(rec_table)


# 11. ID Matching
add_heading("5. TED Talk ID Matching")

matched = len(interaction_ids) - len(unmatched_ids)

add_text(
    "Matched IDs: " + str(matched) +
    "<br/>Unmatched IDs: " + str(len(unmatched_ids))
)

if unmatched_ids:
    add_text("Unmatched IDs: " + escape(", ".join(unmatched_ids)))


# 12. Model Limitations
add_heading("6. Model Limitations")

limitations = [
    "Sparse data: many users have not rated some talks.",
    "Simulated ratings: the data is not from real users.",
    "Cold-start users: new users have no rating history.",
    "Cold-start items: new talks have no ratings.",
    "Zero-filling: missing ratings are treated as zero for similarity calculation.",
    "Popularity bias: popular talks may get more recommendations.",
    "Few common ratings: similarity scores may be unreliable."
]

for item in limitations:
    add_text("• " + escape(item))


# 13. Conclusion
add_heading("7. Conclusion")

add_text(
    "The collaborative filtering model finds users with similar "
    "rating patterns and recommends talks that the target user "
    "has not rated. It uses similarity-weighted ratings to predict "
    "user preferences. This is an initial user-based model. "
    "A hybrid recommendation model can be developed in the future."
)


# 14. Generate PDF
pdf.build(content)

print("PDF generated successfully!")
print("Saved at:", pdf_path)


ModuleNotFoundError: No module named 'reportlab'